# EMASEO -- Entrenamiento RT-DETR-L | Google Colab T4

| | |
|---|---|
| **Modelo** | RT-DETR-L (Real-Time Detection Transformer Large) |
| **Objetivo** | mAP@50 >= 0.85 |
| **Baseline actual** | mAP@50 = 0.4752 |
| **Dataset** | 21,987 train / 3,531 valid / nc=1 (garbage) |

---

## ANTES DE EJECUTAR
1. **Runtime > Change runtime type > GPU T4 > Guardar**
2. La carpeta `dataset_entrenamiento` debe estar en `Mi unidad/` en Google Drive
3. Ejecuta las celdas en orden

---

## FLUJO
```
Primera vez:     Celda 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7
Si se reconecta: Celda 1 -> 2 -> 3 -> 4 -> 8 (resume rapido)
Al terminar:     Celda 9 (validar) -> Celda 10 (descargar)
```

---

## ANTI-IDLE (pegar en consola del navegador F12 antes de entrenar)
```javascript
setInterval(function() {
    document.dispatchEvent(new MouseEvent('mousemove', {
        view: window, bubbles: true, cancelable: true,
        clientX: Math.random() * window.innerWidth,
        clientY: Math.random() * window.innerHeight
    }));
    console.log('Anti-idle: ' + new Date().toLocaleTimeString());
}, 55000)
```

---
## CELDA 1 -- Verificar GPU
Confirma que tienes T4 con ~15 GB VRAM antes de continuar.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No hay GPU. Ve a Runtime > Change runtime type > GPU T4')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'GPU:  {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')

if 'T4' not in gpu_name:
    print('AVISO: No es T4. El entrenamiento puede ser mas lento o fallar por VRAM.')
else:
    print('OK -- T4 lista')

---
## CELDA 2 -- Instalar ultralytics
Tarda ~2 minutos. Necesaria cada vez que Colab se reconecta.

In [ ]:
!pip install ultralytics==8.3.* -q

from ultralytics import RTDETR
import ultralytics
print(f'ultralytics {ultralytics.__version__} instalado OK')

---
## CELDA 3 -- Montar Drive
Solo monta Drive. No extrae ningun archivo.
Necesaria cada vez que Colab se reconecta.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DRIVE_DATASET = '/content/drive/MyDrive/dataset_entrenamiento'
DRIVE_RUNS    = '/content/drive/MyDrive/emaseo_runs'

if not Path(DRIVE_DATASET).exists():
    raise FileNotFoundError(
        f'No se encontro {DRIVE_DATASET}\n'
        'Sube la carpeta dataset_entrenamiento a Mi unidad/ en Google Drive'
    )

print(f'Drive montado OK')
print(f'Dataset en Drive: {DRIVE_DATASET}')

---
## CELDA 4 -- Copiar dataset a disco local (FIX DE VELOCIDAD)

> **Por que esta celda existe:**
> Leer imagenes directo desde Drive tiene ~400ms de latencia por archivo.
> Con 21,987 imagenes, cada epoca tarda 3-4 horas.
> Al copiar al disco local de Colab (SSD), la epoca baja a ~25-35 minutos.
>
> **Tiempo:** ~20-30 minutos (solo la primera vez por sesion).
> Usa 16 hilos en paralelo para copiar rapido.
>
> El resultado se guarda en `/content/dataset_local/`

In [ ]:
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import time

DRIVE_DATASET = Path('/content/drive/MyDrive/dataset_entrenamiento')
LOCAL_DATASET = Path('/content/dataset_local')

if LOCAL_DATASET.exists():
    imgs = list((LOCAL_DATASET / 'train' / 'images').glob('*'))
    print(f'Dataset ya esta en disco local: {len(imgs):,} imagenes en train')
else:
    print('Copiando dataset a disco local con 16 hilos paralelos...')
    print('Esto tarda ~20-30 min una sola vez por sesion.')
    print()

    # Listar todos los archivos a copiar
    files = [(f, LOCAL_DATASET / f.relative_to(DRIVE_DATASET))
             for f in DRIVE_DATASET.rglob('*') if f.is_file()]

    print(f'Total archivos: {len(files):,}')

    def copy_one(args):
        src, dst = args
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

    t0 = time.time()
    done = 0
    with ThreadPoolExecutor(max_workers=16) as ex:
        for _ in ex.map(copy_one, files):
            done += 1
            if done % 2000 == 0:
                pct = done / len(files) * 100
                elapsed = time.time() - t0
                eta = (elapsed / done) * (len(files) - done)
                print(f'  {done:>6,}/{len(files):,} ({pct:.0f}%)  '
                      f'Tiempo: {elapsed/60:.1f} min  '
                      f'Restante: ~{eta/60:.0f} min')

    total = time.time() - t0
    print(f'\nCopia completada en {total/60:.1f} minutos')

# Verificar
IMG_EXTS = {'.jpg', '.jpeg', '.png'}
for split in ['train', 'valid']:
    imgs_dir   = LOCAL_DATASET / split / 'images'
    labels_dir = LOCAL_DATASET / split / 'labels'
    imgs   = [f for f in imgs_dir.iterdir() if f.suffix in IMG_EXTS] if imgs_dir.exists() else []
    labels = list(labels_dir.glob('*.txt')) if labels_dir.exists() else []
    print(f'{split:5}: {len(imgs):>6,} imagenes | {len(labels):>6,} labels | {len(imgs)-len(labels):>4} backgrounds')

print('\nDataset local listo OK')

---
## CELDA 5 -- Configurar data.yaml local
Escribe el data.yaml apuntando al disco local de Colab.
Ejecutar despues de la Celda 4.

In [ ]:
from pathlib import Path

LOCAL_DATASET = Path('/content/dataset_local')

yaml_content = f"""path: {LOCAL_DATASET}
train: train/images
val:   valid/images
test:  test/images
nc: 1
names:
  - garbage
"""

yaml_path = LOCAL_DATASET / 'data.yaml'
yaml_path.write_text(yaml_content)

print('data.yaml configurado:')
print(yaml_path.read_text())

---
## CELDA 6 -- Verificar espacio en disco
cache='disk' necesita ~15 GB libres. Verificar antes de entrenar.

In [ ]:
import shutil

total, used, free = shutil.disk_usage('/content')
print(f'Disco /content:')
print(f'  Total: {total/1e9:.1f} GB')
print(f'  Usado: {used/1e9:.1f} GB')
print(f'  Libre: {free/1e9:.1f} GB')

if free / 1e9 < 15:
    print('AVISO: Menos de 15 GB libres. cache=disk puede fallar.')
    print('Cambia cache="disk" por cache=False en las celdas 7 y 8.')
else:
    print('OK -- espacio suficiente para cache=disk')

---
## CELDA 7 -- ENTRENAMIENTO (primera vez)

> Ejecutar SOLO la primera vez. Si Colab se desconecta, usar **Celda 8** para resumir.
>
> **Velocidad esperada:** ~25-35 min/epoca desde disco local.
>
> El modelo se guarda en Drive cada epoca automaticamente.
>
> **Anti-idle:** Pega el JavaScript del encabezado en la consola del navegador (F12)
> ANTES de ejecutar esta celda.

In [ ]:
from ultralytics import RTDETR

LOCAL_DATASET = '/content/dataset_local'

model = RTDETR('rtdetr-l.pt')  # descarga pesos COCO base (~2 min)

results = model.train(
    # Dataset
    data=f'{LOCAL_DATASET}/data.yaml',
    imgsz=640,
    cache='disk',      # SSD local ~500 MB/s >> Drive ~1-5 MB/s efectivo

    # Entrenamiento
    epochs=100,
    batch=16,
    device=0,
    workers=4,
    amp=True,

    # Optimizador
    optimizer='AdamW',
    lr0=0.0001,
    lrf=0.01,
    warmup_epochs=5,
    patience=25,

    # Augmentacion
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    copy_paste=0.1,

    # Guardado en Drive
    project='/content/drive/MyDrive/emaseo_runs',
    name='rtdetr_l_emaseo_v2',
    exist_ok=True,
    save=True,
    save_period=1,     # checkpoint CADA epoca -- maximo 25-35 min perdidos
    verbose=True,
)

map50 = results.results_dict.get('metrics/mAP50(B)', 0)
print(f'\nmAP@50 final:   {map50:.4f}')
print(f'Baseline previo: 0.4752')
print(f'Mejora:          {map50 - 0.4752:+.4f}')

---
## CELDA 8 -- REANUDAR entrenamiento (si Colab se desconecto)

> **Orden obligatorio antes de esta celda:**
> Celda 1 -> Celda 2 -> Celda 3 -> Celda 4 -> **Celda 8**
>
> NO ejecutes Celda 7 -- eso reinicia desde cero.
>
> Esta celda parchea el checkpoint para usar el dataset local
> y luego reanuda exactamente donde quedo (misma epoca, mismo lr).

In [ ]:
import torch
from pathlib import Path
from ultralytics import RTDETR

LAST_PT    = '/content/drive/MyDrive/emaseo_runs/rtdetr_l_emaseo_v2/weights/last.pt'
LOCAL_DATA = '/content/dataset_local/data.yaml'

if not Path(LAST_PT).exists():
    raise FileNotFoundError(
        f'No se encontro {LAST_PT}\n'
        'Verifica que Drive esta montado (Celda 3) y que el dataset esta copiado (Celda 4).'
    )

if not Path(LOCAL_DATA).exists():
    raise FileNotFoundError(
        'No se encontro data.yaml local.\n'
        'Ejecuta la Celda 4 (copiar dataset) y la Celda 5 (data.yaml) primero.'
    )

# Parchear checkpoint para usar dataset local y cache=disk
# weights_only=False requerido desde PyTorch 2.6
print('Parcheando checkpoint para usar disco local...')
ckpt = torch.load(LAST_PT, map_location='cpu', weights_only=False)
epoch_actual = ckpt.get('epoch', '?')
print(f'  Checkpoint en epoca: {epoch_actual}')

if 'train_args' in ckpt:
    ckpt['train_args']['data']        = LOCAL_DATA
    ckpt['train_args']['cache']       = 'disk'
    ckpt['train_args']['workers']     = 4
    ckpt['train_args']['save_period'] = 1
    torch.save(ckpt, LAST_PT)
    print('  Checkpoint actualizado OK')
else:
    print('  AVISO: train_args no encontrado -- usando resume sin parche')

print(f'Reanudando desde epoca {epoch_actual}...')

model = RTDETR(LAST_PT)
results = model.train(resume=True)

map50 = results.results_dict.get('metrics/mAP50(B)', 0)
print(f'\nmAP@50 final: {map50:.4f}')

---
## CELDA 9 -- Validacion final con best.pt
Ejecutar cuando el entrenamiento haya terminado (early stopping o 100 epocas).

In [ ]:
from ultralytics import RTDETR
from pathlib import Path

LOCAL_DATASET = '/content/dataset_local'
BEST_PT       = '/content/drive/MyDrive/emaseo_runs/rtdetr_l_emaseo_v2/weights/best.pt'

if not Path(BEST_PT).exists():
    raise FileNotFoundError('No se encontro best.pt. El entrenamiento debe terminar primero.')

model = RTDETR(BEST_PT)

metrics = model.val(
    data=f'{LOCAL_DATASET}/data.yaml',
    imgsz=640,
    batch=16,
    device=0,
)

map50 = metrics.results_dict['metrics/mAP50(B)']
prec  = metrics.results_dict['metrics/precision(B)']
rec   = metrics.results_dict['metrics/recall(B)']

print(f'\n==============================')
print(f'  RESULTADOS FINALES')
print(f'==============================')
print(f'  mAP@50:    {map50:.4f}  (baseline: 0.4752)')
print(f'  Precision: {prec:.4f}')
print(f'  Recall:    {rec:.4f}')
print(f'==============================')

if map50 >= 0.85:
    print('  OBJETIVO ALCANZADO -- listo para produccion')
elif map50 >= 0.75:
    print('  Muy buen resultado. Fine-tuning con fotos de Quito llevara al 0.85')
elif map50 >= 0.65:
    print('  Progreso solido. Ejecuta Celda 8 para mas epocas.')
else:
    print('  Continuar entrenamiento con Celda 8.')

---
## CELDA 10 -- Descargar best.pt
Descarga el modelo final a tu PC.

In [ ]:
from google.colab import files
from pathlib import Path

BEST_PT = '/content/drive/MyDrive/emaseo_runs/rtdetr_l_emaseo_v2/weights/best.pt'

if not Path(BEST_PT).exists():
    raise FileNotFoundError('No se encontro best.pt. Ejecuta Celda 9 primero.')

size_mb = Path(BEST_PT).stat().st_size / 1e6
print(f'Descargando best.pt ({size_mb:.1f} MB)...')
files.download(BEST_PT)
print('Listo. Entrega el archivo best.pt para integrarlo al microservicio.')